# Traffic Sign Recognition (GTSRB)

Task family: image classification.

Dataset source: https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign

## Why Classification Is Correct

GTSRB provides traffic-sign class labels for image samples. This notebook uses YOLO classification mode (YOLO26m-cls preferred with practical fallback) because the task input is class-labeled sign images, not detection boxes.

## Environment Setup

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    pkg = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure_package('kagglehub')
ensure_package('numpy')
ensure_package('pandas')
ensure_package('PIL', 'Pillow')
ensure_package('matplotlib')
ensure_package('sklearn', 'scikit-learn')
ensure_package('ultralytics')
print('Dependencies are ready.')

## Imports and Configuration

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, top_k_accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
WORK_DIR = PROJECT_DIR / 'working' / 'gtsrb'
PREP_DIR = WORK_DIR / 'prepared_cls'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
for d in [WORK_DIR, PREP_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.ppm'}
MAX_TRAIN_PER_CLASS = 500
MAX_VAL_PER_CLASS = 150
MAX_EVAL_SAMPLES = 2500

print(f'Project dir: {PROJECT_DIR}')

## Real Dataset Download

In [ ]:
import kagglehub

if not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):
    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')

dataset_root = Path(kagglehub.dataset_download('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign'))
print(f'Dataset root: {dataset_root}')

## Discover Files, Verify Labels, and Build Train/Val Split

In [ ]:
def first_existing(paths: list[Path]) -> Path | None:
    for p in paths:
        if p.exists():
            return p
    return None

train_root = first_existing([
    dataset_root / 'Train',
    dataset_root / 'train',
    dataset_root / 'Training',
    dataset_root / 'GTSRB' / 'Train'
])
if train_root is None:
    raise FileNotFoundError('Train folder not found in downloaded dataset.')

test_root = first_existing([
    dataset_root / 'Test',
    dataset_root / 'test',
    dataset_root / 'Testing',
    dataset_root / 'GTSRB' / 'Test'
])

test_csv = first_existing([
    dataset_root / 'Test.csv',
    dataset_root / 'test.csv',
    dataset_root / 'GT-final_test.csv'
])

rows = []
class_dirs = [p for p in train_root.iterdir() if p.is_dir()]
if len(class_dirs) < 2:
    raise RuntimeError('Expected multiple class folders inside train root.')

for cls_dir in class_dirs:
    cls_name = cls_dir.name
    image_files = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    if len(image_files) == 0:
        continue
    for img_path in image_files:
        rows.append({'image_path': str(img_path), 'label': str(cls_name)})

df_train_all = pd.DataFrame(rows)
if len(df_train_all) == 0:
    raise RuntimeError('No training images discovered.')

class_list = sorted(df_train_all['label'].unique().tolist())
if len(class_list) < 2:
    raise RuntimeError('Need at least 2 classes for classification.')

train_parts = []
val_parts = []
for cls in class_list:
    part = df_train_all[df_train_all['label'] == cls].copy().sample(frac=1.0, random_state=SEED)
    n = len(part)
    split_idx = max(1, int(0.85 * n))
    tr = part.iloc[:split_idx].copy()
    va = part.iloc[split_idx:].copy()
    if len(va) == 0 and len(tr) > 1:
        va = tr.iloc[-1:].copy()
        tr = tr.iloc[:-1].copy()
    train_parts.append(tr)
    val_parts.append(va)

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)

print(f'Train root: {train_root}')
print(f'Test root: {test_root}')
print(f'Test CSV: {test_csv}')
print(f'Total discovered train rows: {len(df_train_all)}')
print(f'Classes: {len(class_list)}')
print(f'Train rows: {len(train_df)} | Val rows: {len(val_df)}')
train_df['label'].value_counts().sort_index().head()

In [ ]:
sample_paths = train_df['image_path'].sample(n=min(500, len(train_df)), random_state=SEED).tolist()
bad = 0
for p in sample_paths:
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception:
        bad += 1
if bad > 0:
    raise RuntimeError(f'Corrupted image files in sampled integrity check: {bad}')
print('Sampled image integrity check passed.')

## Prepare YOLO Classification Directory

In [ ]:
def cap_per_class(df: pd.DataFrame, limit: int) -> pd.DataFrame:
    pieces = []
    for cls in class_list:
        chunk = df[df['label'] == cls].copy()
        if len(chunk) > limit:
            chunk = chunk.sample(n=limit, random_state=SEED)
        pieces.append(chunk)
    return pd.concat(pieces, ignore_index=True)

train_small = cap_per_class(train_df, MAX_TRAIN_PER_CLASS)
val_small = cap_per_class(val_df, MAX_VAL_PER_CLASS)

for split_name in ['train', 'val']:
    for cls in class_list:
        (PREP_DIR / split_name / cls).mkdir(parents=True, exist_ok=True)

for _, row in train_small.iterrows():
    src = Path(row['image_path'])
    dst = PREP_DIR / 'train' / row['label'] / src.name
    if not dst.exists():
        shutil.copy2(src, dst)

for _, row in val_small.iterrows():
    src = Path(row['image_path'])
    dst = PREP_DIR / 'val' / row['label'] / src.name
    if not dst.exists():
        shutil.copy2(src, dst)

print(f'Prepared train rows: {len(train_small)}')
print(f'Prepared val rows: {len(val_small)}')

In [ ]:
def count_images(root: Path) -> int:
    return sum(1 for p in root.rglob('*') if p.suffix.lower() in IMAGE_EXTS)

n_train = count_images(PREP_DIR / 'train')
n_val = count_images(PREP_DIR / 'val')
if n_train == 0 or n_val == 0:
    raise RuntimeError('Prepared train/val counts are zero.')

print(f'Train image files: {n_train}')
print(f'Val image files: {n_val}')

## Real Data Visualization

In [ ]:
viz = train_small.sample(n=min(9, len(train_small)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(3, 3, figsize=(11, 10))
for i in range(len(viz)):
    row = viz.iloc[i]
    img = Image.open(row['image_path']).convert('RGB')
    axes.flatten()[i].imshow(img)
    axes.flatten()[i].set_title(row['label'])
    axes.flatten()[i].axis('off')
for j in range(len(viz), 9):
    axes.flatten()[j].axis('off')
plt.tight_layout()
sample_grid_path = ARTIFACTS_DIR / 'sample_grid.png'
plt.savefig(sample_grid_path, dpi=140)
plt.show()

## Train YOLO Classification Model

In [ ]:
from ultralytics import YOLO

weights_candidates = ['yolo26m-cls.pt', 'yolo11m-cls.pt', 'yolov8m-cls.pt']
selected_weights = None
model = None
for w in weights_candidates:
    try:
        model = YOLO(w)
        selected_weights = w
        break
    except Exception:
        continue

if selected_weights is None:
    raise RuntimeError('No classification checkpoint could be loaded.')

print(f'Selected checkpoint: {selected_weights}')

_ = model.train(
    data=str(PREP_DIR),
    epochs=2,
    imgsz=96,
    batch=128,
    project=str(ARTIFACTS_DIR / 'yolo_runs'),
    name='traffic_sign_cls',
    seed=SEED,
    verbose=False
)

best_model_path = Path(model.trainer.best)
print(f'Best model path: {best_model_path}')

## Real Validation Evaluation

In [ ]:
best_model = YOLO(str(best_model_path))

ordered_labels = sorted(class_list)
label_to_idx = {name: i for i, name in enumerate(ordered_labels)}
idx_to_label = {v: k for k, v in label_to_idx.items()}

eval_df = val_small.copy().reset_index(drop=True)
if len(eval_df) > MAX_EVAL_SAMPLES:
    eval_df = eval_df.sample(n=MAX_EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)

y_true = []
y_pred = []
y_prob_rows = []

for _, row in eval_df.iterrows():
    pred = best_model.predict(source=row['image_path'], verbose=False)[0]
    probs = pred.probs.data.cpu().numpy()
    y_true.append(label_to_idx[row['label']])
    y_pred.append(int(np.argmax(probs)))
    y_prob_rows.append(probs)

y_prob = np.vstack(y_prob_rows)

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
top3 = top_k_accuracy_score(y_true, y_prob, k=3, labels=list(range(len(ordered_labels))))
cm = confusion_matrix(y_true, y_pred)

print(f'Eval samples: {len(eval_df)}')
print(f'Accuracy: {acc:.4f}')
print(f'Macro F1: {macro_f1:.4f}')
print(f'Top-3 accuracy: {top3:.4f}')
print(classification_report(y_true, y_pred, labels=list(range(len(ordered_labels))), target_names=ordered_labels, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm, cmap='Blues')
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ticks = list(range(len(ordered_labels)))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(ordered_labels, rotation=90, fontsize=7)
ax.set_yticklabels(ordered_labels, fontsize=7)
plt.tight_layout()
cm_path = ARTIFACTS_DIR / 'confusion_matrix.png'
plt.savefig(cm_path, dpi=140)
plt.show()

## Honest Qualitative Error Review

In [ ]:
pred_df = eval_df.copy()
pred_df['true_label'] = [idx_to_label[i] for i in y_true]
pred_df['pred_label'] = [idx_to_label[i] for i in y_pred]
pred_df['correct'] = (pred_df['true_label'] == pred_df['pred_label']).astype(int)

hard = pred_df[pred_df['correct'] == 0].head(9)
if len(hard) == 0:
    hard = pred_df.sample(n=min(9, len(pred_df)), random_state=SEED)

fig, axes = plt.subplots(3, 3, figsize=(11, 10))
for i in range(len(hard)):
    row = hard.iloc[i]
    img = Image.open(row['image_path']).convert('RGB')
    axes.flatten()[i].imshow(img)
    axes.flatten()[i].set_title(f"t={row['true_label']} p={row['pred_label']}")
    axes.flatten()[i].axis('off')
for j in range(len(hard), 9):
    axes.flatten()[j].axis('off')
plt.tight_layout()
qual_path = ARTIFACTS_DIR / 'qualitative_errors.png'
plt.savefig(qual_path, dpi=140)
plt.show()

## Save Real Outputs

In [ ]:
pred_csv = ARTIFACTS_DIR / 'eval_predictions.csv'
pred_df.to_csv(pred_csv, index=False)

metrics = {
    'dataset': 'meowmeowmeowmeowmeow/gtsrb-german-traffic-sign',
    'dataset_url': 'https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign',
    'train_root': str(train_root),
    'test_root': str(test_root) if test_root is not None else None,
    'test_csv': str(test_csv) if test_csv is not None else None,
    'num_classes': int(len(ordered_labels)),
    'rows_train_original': int(len(train_df)),
    'rows_val_original': int(len(val_df)),
    'rows_train_prepared': int(len(train_small)),
    'rows_val_prepared': int(len(val_small)),
    'selected_weights': selected_weights,
    'best_model_path': str(best_model_path),
    'eval_samples': int(len(eval_df)),
    'accuracy': float(acc),
    'macro_f1': float(macro_f1),
    'top3_accuracy': float(top3)
}

metrics_path = ARTIFACTS_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

manifest = {
    'sample_grid_png': str(sample_grid_path),
    'confusion_matrix_png': str(cm_path),
    'qualitative_png': str(qual_path),
    'predictions_csv': str(pred_csv),
    'metrics_json': str(metrics_path),
    'prepared_data_root': str(PREP_DIR)
}
manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved outputs:')
print(metrics_path)
print(pred_csv)
print(sample_grid_path)
print(cm_path)
print(qual_path)
print(manifest_path)

## Limitations

- This notebook uses per-class row caps and short training to stay practical on local hardware.
- For stronger benchmark performance, train longer and tune augmentation/image size.
- This is an educational workflow and not a deployed safety-critical system.